# 🚀 SPARKY Colab GPU 세팅
런타임 재시작 시 아래 셀 하나만 실행하면 됩니다.

In [ ]:
import subprocess, os, time, requests

NGROK_TOKEN  = "2AKQSbBRsKXiDNbrTziKJubmn2X_ivAbsnjZEyjveS1bSqw5"
OLLAMA_MODEL = "gemma4:e4b"

# ── Step 1: GPU 확인 ─────────────────────────────────
print("🔍 GPU 확인 중...")
result = subprocess.run(["/usr/bin/nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✅ GPU: {result.stdout.strip()}")
else:
    print("❌ GPU 없음 → 런타임 유형을 A100으로 변경하세요")
    raise SystemExit

# ── Step 2: zstd + Ollama 설치 ───────────────────────
print("\n📦 zstd 설치 중...")
os.system("apt-get install -y zstd -q")

print("📦 Ollama 설치 중...")
os.system("curl -fsSL https://ollama.com/install.sh | sh")

# ── Step 3: Ollama 서버 실행 (GPU + 외부 접속 허용) ──
print("\n🚀 Ollama 서버 시작...")
env = os.environ.copy()
env["OLLAMA_ORIGINS"] = "*"
env["OLLAMA_HOST"]    = "0.0.0.0:11434"
subprocess.Popen(["/usr/local/bin/ollama", "serve"], env=env)
time.sleep(5)

try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("✅ Ollama 서버 기동 완료")
except:
    print("⚠️ 서버 응답 느림 → 5초 추가 대기")
    time.sleep(5)

# ── Step 4: 모델 pull ────────────────────────────────
print(f"\n📥 모델 pull 중: {OLLAMA_MODEL}")
os.system(f"ollama pull {OLLAMA_MODEL}")

r = requests.get("http://localhost:11434/api/tags")
models = [m["name"] for m in r.json().get("models", [])]
if any(OLLAMA_MODEL in m for m in models):
    print(f"✅ 모델 준비 완료: {OLLAMA_MODEL}")
else:
    print(f"❌ 모델 없음: {models}")

# ── Step 5: ngrok 터널 ───────────────────────────────
print("\n🌐 ngrok 터널 연결 중...")
os.system("pip install pyngrok -q")
from pyngrok import ngrok

for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(11434)
time.sleep(2)

try:
    r = requests.get(
        f"{public_url.public_url}/api/tags",
        headers={"ngrok-skip-browser-warning": "true"},
        timeout=10
    )
    status = "✅ ngrok 연결 성공" if r.status_code == 200 else f"⚠️ ngrok 응답 이상: {r.status_code}"
    print(status)
except Exception as e:
    print(f"⚠️ ngrok 확인 실패: {e}")

# ── 완료 ─────────────────────────────────────────────
print("\n" + "="*50)
print(f"✅ OLLAMA_BASE_URL = {public_url.public_url}")
print(f"✅ OLLAMA_MODEL    = {OLLAMA_MODEL}")
print("="*50)
print("→ 로컬 .env에 위 URL 붙여넣고 ai-server 재시작하세요")

# ── Step 6: 런타임 유지 ──────────────────────────────
print("\n⏳ 런타임 유지 중... (중단하려면 셀 정지)")
while True:
    time.sleep(30)